# MedXIR query variants: Qwen3-8B rewrites, encoded with Octen-8B and Qwen3-Embedding-8B

Generated by `scripts/build_variants_notebook.py`. Edit the generator, not this file.

Run on RTX Pro 6000 with Internet OFF. Attach the packages of Qwen3-8B, Octen-Embedding-8B and
Qwen3-Embedding-8B (kernel_sources) and `boinhbo/medxir-retrieval-inputs`.

In [ ]:
import faulthandler
import json
import os
import resource
import time
import traceback
from contextlib import contextmanager
from pathlib import Path

OUT = Path('/kaggle/working')
OUT.mkdir(parents=True, exist_ok=True)
T0 = time.perf_counter()
PROGRESS = OUT / 'progress.txt'
FAULT_FILE = open(OUT / 'faulthandler.txt', 'w')  # native crashes dump the Python stack here
faulthandler.enable(file=FAULT_FILE, all_threads=True)
FAILED = None


def note(message):
    rss_gib = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2 ** 20  # Linux reports KiB
    line = f'{time.perf_counter() - T0:8.1f}s  peak_rss={rss_gib:.1f}GiB  {message}'
    print(line, flush=True)
    with open(PROGRESS, 'a', encoding='utf-8') as file:
        file.write(line + '\n')


@contextmanager
def stage(name):
    # A Python exception is written to retrieval_error.txt and does not stop the kernel, so the
    # session ends COMPLETE and the files can be fetched; later steps are skipped.
    global FAILED
    note(f'start {name}')
    try:
        yield
        note(f'done {name}')
    except Exception:
        FAILED = name
        (OUT / 'retrieval_error.txt').write_text(f'stage: {name}\n\n{traceback.format_exc()}', encoding='utf-8')
        note(f'FAILED {name}; traceback in retrieval_error.txt')

## 1. Inputs and GPU check

In [ ]:
if FAILED is None:
    with stage('setup'):
        import glob
        import re
        import subprocess

        import numpy as np
        import pandas as pd
        import torch

        os.environ['HF_HUB_OFFLINE'] = '1'


        def one_match(pattern, description):
            hits = sorted(glob.glob(pattern, recursive=True))
            assert len(hits) == 1, f'{description}: expected one match, found {hits[:10]}'
            return Path(hits[0])


        QUERY_PATH = one_match('/kaggle/input/**/query.parquet', 'query.parquet')
        LLM_PATH = one_match('/kaggle/input/**/Qwen3-8B/config.json', 'packaged Qwen3-8B').parent
        EMBEDDERS = {name: one_match(f'/kaggle/input/**/{folder}/config.json', folder).parent
                     for name, folder in [('octen8b', 'Octen-Embedding-8B'), ('qwen8b', 'Qwen3-Embedding-8B')]}
        DOC_PREFIX = {'octen8b': ' ', 'qwen8b': ''}  # as in the corpus embedding kernels
        queries = pd.read_parquet(QUERY_PATH, columns=['id', 'query'])
        assert len(queries) == 1200 and queries['id'].is_unique
        print('llm:', LLM_PATH, '| embedders:', EMBEDDERS)

        print(subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total,driver_version', '--format=csv'],
                             capture_output=True, text=True).stdout)
        gpu_name = torch.cuda.get_device_name(0)
        assert '6000' in gpu_name, f'Expected RTX Pro 6000, got {gpu_name!r}'
        print('GPU:', gpu_name)

## 2. Generate the variants with Qwen3-8B

In [ ]:
if FAILED is None:
    with stage('generate'):
        from transformers import AutoModelForCausalLM, AutoTokenizer

        GEN_BATCH = 48
        MAX_NEW_TOKENS = 448
        SYSTEM = """Bạn là bác sĩ hỗ trợ tìm tài liệu y khoa tiếng Việt. Với câu hỏi của người bệnh, trả về đúng một đối tượng JSON có 3 khóa:
        - "question": viết lại thành MỘT câu hỏi tiếng Việt có dấu, đúng chính tả, tối đa 30 từ. Giữ triệu chứng, vị trí, thời gian, tuổi, giới tính, bệnh đã được chẩn đoán, thuốc và xét nghiệm đã nêu. Bỏ lời chào, lời cảm ơn và phần kể chuyện không cần cho chuyên môn. Không thêm thông tin mới.
        - "terms": 3 đến 8 thuật ngữ y khoa tiếng Việt, phân cách bằng dấu chấm phẩy: tên bệnh được nêu hoặc khả năng cao nhất (kèm tên tiếng Anh hoặc viết tắt trong ngoặc nếu thông dụng), hoạt chất thuốc, xét nghiệm, cơ quan liên quan.
        - "passage": một đoạn 80 đến 120 từ theo văn phong bài viết sức khỏe trên website bệnh viện Việt Nam, trả lời trực tiếp câu hỏi: nguyên nhân thường gặp, dấu hiệu, cách chẩn đoán và hướng xử trí.
        Chỉ trả về JSON, không giải thích."""
        EXAMPLE_Q = 'Con em 3 tuổi mấy hôm nay sốt cao 39 độ, nổi mấy nốt đỏ ở lòng bàn tay với trong miệng, bé bỏ ăn quấy khóc, có sao không ạ?'
        EXAMPLE_A = json.dumps(dict(
            question='Trẻ 3 tuổi sốt cao 39 độ, nổi nốt đỏ ở lòng bàn tay và trong miệng, bỏ ăn là bệnh gì?',
            terms='bệnh tay chân miệng (hand, foot and mouth disease); enterovirus 71 (EV71); sốt cao ở trẻ em; loét miệng; phát ban',
            passage=('Trẻ nhỏ sốt cao kèm nốt đỏ hoặc bọng nước ở lòng bàn tay, bàn chân và vết loét trong miệng là dấu hiệu điển hình '
                     'của bệnh tay chân miệng, do virus đường ruột như Coxsackie A16 và Enterovirus 71 gây ra. Trẻ thường đau miệng nên '
                     'bỏ ăn, quấy khóc, chảy nước bọt. Bác sĩ chẩn đoán chủ yếu dựa vào triệu chứng lâm sàng. Phần lớn trường hợp tự khỏi '
                     'sau 7 đến 10 ngày; cha mẹ cần hạ sốt bằng paracetamol, cho trẻ ăn thức ăn lỏng, mát và bù đủ nước. Cần đưa trẻ đi '
                     'viện ngay nếu sốt cao trên 39 độ không hạ, giật mình, nôn nhiều, li bì hoặc co giật vì có thể là biến chứng thần kinh.'),
        ), ensure_ascii=False)

        tok = AutoTokenizer.from_pretrained(str(LLM_PATH))
        tok.padding_side = 'left'
        llm = AutoModelForCausalLM.from_pretrained(str(LLM_PATH), torch_dtype=torch.bfloat16, attn_implementation='sdpa').cuda().eval()


        def prompt(query):
            messages = [dict(role='system', content=SYSTEM), dict(role='user', content=EXAMPLE_Q),
                        dict(role='assistant', content=EXAMPLE_A), dict(role='user', content=query)]
            return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)


        def parse(text):
            """The first {...} object with the three string fields, or None."""
            text = re.sub(r'<think>.*?</think>', '', text, flags=re.S)
            match = re.search(r'\{.*\}', text, flags=re.S)
            if not match:
                return None
            try:
                value = json.loads(match.group(0))
            except json.JSONDecodeError:
                return None
            fields = {k: value.get(k) for k in ('question', 'terms', 'passage')}
            if not all(isinstance(v, str) and v.strip() for v in fields.values()):
                return None
            return {k: v.strip() for k, v in fields.items()}


        texts = [prompt(q) for q in queries['query'].astype(str)]
        lengths = [len(ids) for ids in tok(texts)['input_ids']]
        order = np.argsort([-n for n in lengths], kind='stable')
        raw = [None] * len(texts)
        with torch.inference_mode():
            for start in range(0, len(order), GEN_BATCH):
                index = order[start:start + GEN_BATCH]
                batch = tok([texts[i] for i in index], return_tensors='pt', padding=True).to('cuda')
                out = llm.generate(**batch, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, temperature=None, top_p=None,
                                   top_k=None, pad_token_id=tok.pad_token_id)
                decoded = tok.batch_decode(out[:, batch['input_ids'].shape[1]:], skip_special_tokens=True)
                for i, text in zip(index, decoded):
                    raw[i] = text
                note(f'generated {min(start + GEN_BATCH, len(order))}/{len(order)}')
        del llm
        torch.cuda.empty_cache()

        rows = []
        for (qid, query), text in zip(queries.itertuples(index=False), raw):
            fields = parse(text)
            rows.append(dict(id=int(qid), parsed=fields is not None, raw=text,
                             **(fields or dict(question=query, terms=query, passage=query))))
        variants = pd.DataFrame(rows)
        variants.to_parquet(OUT / 'variants.parquet', index=False)
        print('parsed', int(variants['parsed'].sum()), '/', len(variants))
        print('words, median:', {k: float(variants[k].str.split().str.len().median()) for k in ('question', 'terms', 'passage')})
        for row in variants.head(3).itertuples():
            print(row.id, '|', row.question, '|', row.terms, '|', row.passage[:200])
        assert variants['parsed'].mean() > 0.9, 'more than 10% of the outputs did not parse'

## 3. Encode the variants with both 8B embedders

In [ ]:
if FAILED is None:
    with stage('encode'):
        from transformers import AutoModel, AutoTokenizer
        import torch.nn.functional as F

        TASK = 'Given a web search query, retrieve relevant passages that answer the query'
        MAX_LENGTH = 1024
        EMBED_BATCH = 16


        def last_token_pool(hidden, mask):
            if mask[:, -1].sum() == mask.shape[0]:
                return hidden[:, -1]
            lengths = mask.sum(dim=1) - 1
            return hidden[torch.arange(hidden.shape[0], device=hidden.device), lengths]


        @torch.inference_mode()
        def encode(model, tok, texts):
            tokenized = tok(texts, truncation=True, max_length=MAX_LENGTH)['input_ids']
            order = np.argsort([-len(row) for row in tokenized], kind='stable')
            vectors = np.empty((len(texts), model.config.hidden_size), dtype=np.float32)
            for start in range(0, len(order), EMBED_BATCH):
                index = order[start:start + EMBED_BATCH]
                batch = tok.pad({'input_ids': [tokenized[i] for i in index]}, return_tensors='pt').to('cuda')
                hidden = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'],
                               use_cache=False).last_hidden_state
                vectors[index] = F.normalize(last_token_pool(hidden, batch['attention_mask']).float(), p=2, dim=1).cpu().numpy()
            return vectors


        variants = pd.read_parquet(OUT / 'variants.parquet')
        assert variants['id'].tolist() == queries['id'].tolist()
        for name, path in EMBEDDERS.items():
            tok = AutoTokenizer.from_pretrained(str(path))
            tok.padding_side = 'left'
            model = AutoModel.from_pretrained(str(path), torch_dtype=torch.bfloat16, attn_implementation='sdpa').cuda().eval()
            for field in ('question', 'terms', 'passage'):
                if field == 'passage':
                    texts = [DOC_PREFIX[name] + t for t in variants[field]]
                else:
                    texts = [f'Instruct: {TASK}\nQuery:{t}' for t in variants[field]]
                vectors = encode(model, tok, texts)
                assert vectors.shape == (1200, 4096) and np.isfinite(vectors).all()
                np.save(OUT / f'vectors-{name}-{field}.npy', vectors)
                note(f'encoded {name} {field}')
            del model, tok
            torch.cuda.empty_cache()
        (OUT / 'query_ids.json').write_text(json.dumps(queries['id'].tolist()) + '\n', encoding='utf-8')

## 4. Report

In [ ]:
note('finished; status ' + ('error' if FAILED else 'ok'))
if FAILED:
    (OUT / 'stage_status.json').write_text(json.dumps(
        dict(status='error', failed_stage=FAILED, error_file='retrieval_error.txt'), indent=2) + '\n')